# 07 — Libdevice

**Ladder:** Naive `math.erf` loop → vectorized `torch.erf` → Triton `libdevice.erf` → accuracy/perf tradeoff vs pure-`tl` tanh / polys.

**Study tip.** After running cells: write erf-GELU; say when a tanh poly beats libdevice; note that huge-$N$ GELU is usually memory-bound (accuracy often "free").

Official: [Extern functions / libdevice](https://triton-lang.org/main/getting-started/tutorials/07-extern-functions.html)
· [Libdevice slug](https://triton-lang.org/main/getting-started/tutorials/07-libdevice.html)

Figures: [`docs/images/`](docs/images/) · visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

![Whiteboard: pure Triton approx vs libdevice device math](docs/images/benchmark-libdevice.png)

*Theory board — polynomial/tanh approximations vs `tl.extra.libdevice` intrinsics (accuracy / speed tradeoff).*

---

## 0. GELU definition

The Gaussian Error Linear Unit is

$$
\mathrm{GELU}(x)
=
x\,\Phi(x)
=
\tfrac{1}{2}\, x\Big(1 + \mathrm{erf}\big(\tfrac{x}{\sqrt{2}}\big)\Big),
$$

where $\Phi$ is the standard normal CDF and

$$
\mathrm{erf}(z) = \frac{2}{\sqrt{\pi}}\int_0^{z} e^{-t^{2}}\,\mathrm{d}t.
$$

**Constant used in kernels.** $1/\sqrt{2} \approx 0.7071067811865476$.

**Device math.** `libdevice.erf` is the CUDA math-library binding — same formula as `math.erf` / `torch.erf`, GPU-tuned, callable from a Triton elementwise program after `tl.load`.

**Alternate approx (tradeoff rung).**

$$
\mathrm{GELU}_{\tanh}(x)=\tfrac{1}{2}x\Big(1+\tanh\big(\sqrt{2/\pi}\,(x+0.044715\,x^{3})\big)\Big).
$$

Pure `tl` arithmetic; compare ULPs and GB/s against the erf path.

### Theory board

![Whiteboard: pure Triton approx vs libdevice device math](docs/images/whiteboard-theory.jpg)

*Theory board — polynomial/tanh approximations vs `tl.extra.libdevice` intrinsics (accuracy / speed tradeoff).*

### Pitfalls

- libdevice availability differs by backend; check imports.
- Comparing approx tanh-GELU to erf-GELU needs looser tolerances.
- Special functions can use more registers → lower occupancy.

### Self-check

1. Write GELU with erf.
2. When is a pure Triton poly “good enough”?
3. Is a more accurate erf free if the kernel is memory-bound?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import math

import pytest
import torch
import triton
import triton.language as tl
import triton.language.extra.libdevice as libdevice

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



## 1–3. Ladder: Naive → Torch → Triton (`libdevice`)

For each index $i$:

$$
y_i = \tfrac{1}{2} x_i \Big(1 + \mathrm{erf}(x_i/\sqrt{2})\Big).
$$

| Stage | Realization | Notes |
|---|---|---|
| **Naive** | Python loop + `math.erf` | Host reference; obvious formula |
| **Intermediate** | `torch.erf` vectorized | Same math; ATen, not custom Triton |
| **Triton** | `tl.load` → `libdevice.erf` → store | Device special function inside SPMD tile |

**Equivalence.** All three implement the same mathematical map (up to floating-point rounding). Tests compare against `torch.nn.functional.gelu`.

**SPMD tile (Triton rung).** Program $p$ owns offsets $pB+\{0,\ldots,B-1\}$ with mask $\mathrm{offsets}<N$ (same pattern as tutorial 01). Default $B=1024$.

**When to prefer libdevice vs pure Triton.** Need `erf` / awkward specials → libdevice. Already accept a published poly (tanh GELU) and care about every FLOP → pure `tl`. Huge memory-bound $N$ → accuracy often wins for free.


In [ ]:
def naive_gelu_pseudo(x):
    """
    Exact GELU (Gaussian Error Linear Unit):

        gelu(x) = 0.5 * x * (1 + erf(x / sqrt(2)))

    Implemented with a Python loop over elements using math.erf so the
    formula is obvious. Prefer gelu_torch / the Triton kernel for speed.
    """
    flat = x.detach().cpu().reshape(-1)
    out = torch.empty_like(flat)
    inv_sqrt2 = 1.0 / math.sqrt(2.0)
    for i in range(flat.numel()):
        v = float(flat[i])
        out[i] = 0.5 * v * (1.0 + math.erf(v * inv_sqrt2))
    return out.reshape(x.shape).to(device=x.device, dtype=x.dtype)


In [ ]:
def gelu_torch(x):
    """Same formula with vectorized torch ops (still not a custom kernel)."""
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))


In [ ]:
@triton.jit
def gelu_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    """Elementwise GELU using CUDA libdevice erf via Triton's extra bindings."""
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask).to(tl.float32)
    # 0.7071... = 1/sqrt(2); libdevice.erf is the device math library
    y = libdevice.erf(x * 0.7071067811865476)
    y = 0.5 * x * (1.0 + y)
    tl.store(y_ptr + offs, y, mask=mask)


def gelu_libdevice(x):
    y = torch.empty_like(x)
    n = x.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    gelu_kernel[grid](x, y, n, BLOCK_SIZE=1024)
    return y


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    x = torch.randn(128)
    ref = torch.nn.functional.gelu(x)
    assert torch.allclose(naive_gelu_pseudo(x), ref, atol=1e-5, rtol=1e-5)
    assert torch.allclose(gelu_torch(x), ref, atol=1e-5, rtol=1e-5)


def test_correctness() -> None:
    require_gpu()
    x = torch.randn(8192, device=DEVICE, dtype=torch.float32)
    y = gelu_libdevice(x)
    y_ref = torch.nn.functional.gelu(x)
    assert torch.allclose(y, y_ref, atol=1e-2, rtol=1e-2)
    print("gelu via libdevice OK")
